In [1]:
%pip install biopython

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [6]:
from Bio import Entrez, SeqIO, Align



def get_cds(acc):
  rec = SeqIO. read(Entrez.efetch(db="nucleotide", id= acc,
                                  rettype= " fasta", retmode = " text"),"fasta")
  seq = rec.seq
  start = seq.find("ATG") # first start codon
  protein = seq[start:].translate(to_stop=True)
  cds = seq[start:start + 3 * len(protein)]
  return cds, protein

human_cds, human_prot = get_cds("NM_000518")
mouse_cds, mouse_prot = get_cds("NM_008220")

print("Human CDS:", len(human_cds), "codon | protein:", len(human_prot), "protein")
print("Mouse CDS:", len(mouse_cds), "codon | protein:", len(mouse_prot), "protein")
print("Human protein starts:", human_prot[:20])
print("Mouse protein starts:", mouse_prot[:20])




aligner = Align.PairwiseAligner()


aligner.mode = "global"
aln = aligner.align(human_prot, mouse_prot)[0]
print(aln)
matches = sum(a == b for a, b in zip(aln[0], aln[1]) if a != "-" and b != "-")
print("Protein identity:", round(100 * matches / len(aln[0]), 1), "%")

Human CDS: 441 codon | protein: 147 protein
Mouse CDS: 441 codon | protein: 147 protein
Human protein starts: MVHLTPEEKSAVTALWGKVN
Mouse protein starts: MVHLTDAEKAAVSGLWGKVN
target            0 MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPK
                  0 |||||..||.||..||||||.||||||||||||||||||||.|.||||||...|.|||.|
query             0 MVHLTDAEKAAVSGLWGKVNADEVGGEALGRLLVVYPWTQRYFDSFGDLSSASAIMGNAK

target           60 VKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFG
                 60 ||||||||..||.|||.|||.||||||.|||||||||||||||||||||..|.||.||.|
query            60 VKAHGKKVITAFNDGLNHLDSLKGTFASLSELHCDKLHVDPENFRLLGNMIVIVLGHHLG

target          120 KEFTPPVQAAYQKVVAGVANALAHKYH 147
                120 |.|||..|||.||||||||.||||||| 147
query           120 KDFTPAAQAAFQKVVAGVAAALAHKYH 147

Protein identity: 80.3 %
